# Planning Pattern - Plan, Observe, Replan


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

Planning is about breaking a larger forensic task into ordered steps, checking whether the steps still make sense as new observations appear, and revising the plan when the evidence changes the path.

This is the **fourth pattern lab** in the course. The earlier pattern lessons are:

* [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
* [Tool Use Pattern](../lab2_tool_use_pattern/03_lab_notebook.ipynb)
* [ReAct Pattern](../lab3_react_pattern/03_lab_notebook.ipynb)


## Relevant Imports and LLM Client


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


## Build the Artifact Bundles

We will intentionally hide the network-status artifact at first so the initial plan has to be revised when a new observation appears.


In [ ]:
def excerpt(filename: str, n_lines: int = 8) -> str:
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])


partial_artifact_bundle = f"""
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

unlock_events.csv
{excerpt('unlock_events.csv')}

call_log.csv
{excerpt('call_log.csv')}

whatsapp_events.csv
{excerpt('whatsapp_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
"""

new_observation = f"""
network_status.csv
{excerpt('network_status.csv')}
"""

full_artifact_bundle = partial_artifact_bundle + '\n\n' + new_observation


## Part 1: Build the Planning Workflow Manually


In [ ]:
PLANNING_SYSTEM_PROMPT = """
You are a digital forensics planning assistant.
Build ordered investigation plans, update them when new observations expose missing dependencies, and avoid unsupported conclusions.
"""

case_question = (
    'Analyze this mobile forensic timeline-reconstruction case. ' 
    'Create an ordered investigation plan before making a final timing conclusion.'
)

initial_plan_request = (
    'Build an initial investigation plan for this case.\n'
    'Return: (1) investigation goal, (2) ordered steps, (3) evidence needed, ' 
    '(4) replanning triggers.\n\n'
    f'Task:\n{case_question}\n\n'
    f'Artifacts:\n{partial_artifact_bundle}'
)

initial_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': initial_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

print(initial_plan)


The new network observation changes whether WhatsApp delivery could have happened inside the incident window, so the next step is to revise the original plan instead of continuing unchanged.


In [ ]:
revised_plan_request = (
    'Revise the current investigation plan using the new observation below.\n'
    'Return: (1) what changed, (2) revised ordered steps, (3) updated evidence priorities, ' 
    '(4) remaining replanning triggers.\n\n'
    f'Original task:\n{case_question}\n\n'
    f'Current plan:\n{initial_plan}\n\n'
    f'New observation:\n{new_observation}'
)

revised_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': revised_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

print(revised_plan)


Now use the revised plan and the full artifact bundle to produce a bounded timeline conclusion.


In [ ]:
final_report_request = (
    'Using the revised plan and full artifact set, produce a short evidence-cited timeline conclusion.\n'
    'Return: (1) reconstructed timeline, (2) timing conclusion, (3) evidence mapping, ' 
    '(4) what remains uncertain.\n\n'
    f'Revised plan:\n{revised_plan}\n\n'
    f'Artifacts:\n{full_artifact_bundle}'
)

final_timeline_conclusion = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': final_report_request},
    ],
    model=MODEL,
).choices[0].message.content

print(final_timeline_conclusion)


## Part 2: Use the Local `PlanningAgent`

The packaged `PlanningAgent` exposes the same workflow more directly: build an initial plan, revise the plan with new observations, and optionally run a self-reviewed planning pass.


In [ ]:
from agentic_patterns.planning_pattern.planning_agent import PlanningAgent

planning_agent = PlanningAgent(client=client, model=MODEL)


In [ ]:
packaged_initial_plan = planning_agent.build_initial_plan(
    f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}'
)
print(packaged_initial_plan)


In [ ]:
packaged_revised_plan = planning_agent.revise_plan(
    user_msg=f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}',
    current_plan=packaged_initial_plan,
    observations=new_observation,
)
print(packaged_revised_plan)


In [ ]:
planning_agent.run(user_msg=f'{case_question}\n\nArtifacts:\n{full_artifact_bundle}')
